# MINLPLib Smoke Test Results (Table 3)

This notebook reproduces Table 3 from the manuscript: MINLPLib smoke test results.
We load and solve 9 MINLPLib instances via discopt's `.nl` parser and B&B solver,
recording status, objective, time, and node count for each.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

In [2]:
from pathlib import Path

import discopt.modeling as dm
import pandas as pd

In [3]:
# Path to .nl test data
NL_DIR = Path(dm.__file__).resolve().parent.parent.parent / "tests" / "data" / "minlplib"
assert NL_DIR.exists(), f".nl directory not found: {NL_DIR}"
print(f"NL directory: {NL_DIR}")

NL directory: /Users/jkitchin/Dropbox/projects/discopt/python/tests/data/minlplib


In [4]:
# 9 smoke-test instances (matching Table 3 in manuscript)
SMOKE_INSTANCES = [
    {"name": "chance", "best_known": 29.89437816},
    {"name": "dispatch", "best_known": 3155.28792700},
    {"name": "ex1221", "best_known": 7.66718007},
    {"name": "ex1226", "best_known": -17.0},
    {"name": "gear", "best_known": 0.0},
    {"name": "nvs01", "best_known": 12.46966882},
    {"name": "nvs03", "best_known": 16.0},
    {"name": "nvs04", "best_known": 0.72},
    {"name": "nvs06", "best_known": 1.77031250},
]

TIME_LIMIT = 60.0  # seconds per instance

In [5]:
results = []

for inst in SMOKE_INSTANCES:
    name = inst["name"]
    nl_path = str(NL_DIR / f"{name}.nl")
    assert os.path.exists(nl_path), f"Missing .nl file: {nl_path}"

    model = dm.from_nl(nl_path)
    n = model.num_variables
    m = model.num_constraints

    result = model.solve(
        time_limit=TIME_LIMIT,
        gap_tolerance=1e-4,
        max_nodes=100_000,
    )

    obj_val = result.objective
    if obj_val is not None and abs(obj_val) < 1e-8:
        obj_str = f"{obj_val:.1e}"
    elif obj_val is not None:
        obj_str = f"{obj_val:.3f}"
    else:
        obj_str = "--"

    best_known = inst["best_known"]
    if abs(best_known) < 1e-8:
        bk_str = f"{best_known:.3f}"
    else:
        bk_str = f"{best_known:.3f}"

    results.append(
        {
            "Instance": name,
            "n": n,
            "m": m,
            "Status": result.status,
            "f*": obj_str,
            "Best Known": bk_str,
            "Time (s)": f"{result.wall_time:.3f}",
            "Nodes": f"{result.node_count:,}",
        }
    )

    print(
        f"{name:10s}  n={n}  m={m}  status={result.status:10s}  "
        f"obj={obj_str:>12s}  best_known={bk_str:>12s}  "
        f"time={result.wall_time:.3f}s  nodes={result.node_count:,}"
    )

chance      n=4  m=3  status=optimal     obj=      30.317  best_known=      29.894  time=0.927s  nodes=0


dispatch    n=4  m=2  status=optimal     obj=    3166.302  best_known=    3155.288  time=0.871s  nodes=0



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

ex1221      n=5  m=5  status=optimal     obj=       7.667  best_known=       7.667  time=3.822s  nodes=1


ex1226      n=5  m=5  status=optimal     obj=     -17.000  best_known=     -17.000  time=3.334s  nodes=1


gear        n=4  m=0  status=feasible    obj=       0.000  best_known=       0.000  time=68.241s  nodes=233


nvs01       n=3  m=3  status=time_limit  obj=          --  best_known=      12.470  time=67.894s  nodes=69


nvs03       n=2  m=2  status=optimal     obj=      16.000  best_known=      16.000  time=2.967s  nodes=7


nvs04       n=2  m=0  status=optimal     obj=       0.720  best_known=       0.720  time=17.585s  nodes=59


nvs06       n=2  m=0  status=optimal     obj=       1.770  best_known=       1.770  time=5.570s  nodes=9


In [6]:
# Format as table (Table 3 in manuscript)
df = pd.DataFrame(results)
df

,Instance,n,m,Status,f*,Best Known,Time (s),Nodes
0,chance,4,3,optimal,30.317,29.894,0.927,0
1,dispatch,4,2,optimal,3166.302,3155.288,0.871,0
2,ex1221,5,5,optimal,7.667,7.667,3.822,1
3,ex1226,5,5,optimal,-17.000,-17.000,3.334,1
4,gear,4,0,feasible,0.000,0.000,68.241,233
5,nvs01,3,3,time_limit,--,12.470,67.894,69
6,nvs03,2,2,optimal,16.000,16.000,2.967,7
7,nvs04,2,0,optimal,0.720,0.720,17.585,59
8,nvs06,2,0,optimal,1.770,1.770,5.570,9


In [7]:
# LaTeX version for manuscript
print("\\begin{table}[htbp]")
print("\\centering")
print("\\begin{tabular}{llllllll}")
print("\\toprule")
print("Instance & $n$ & $m$ & Status & $f^*$ & Best Known & Time (s) & Nodes \\\\")
print("\\midrule")
for r in results:
    print(
        f"{r['Instance']} & {r['n']} & {r['m']} & {r['Status']} & "
        f"{r['f*']} & {r['Best Known']} & {r['Time (s)']} & {r['Nodes']} \\\\"
    )
print("\\bottomrule")
print("\\end{tabular}")
print(
    "\\caption{MINLPLib smoke test results (9 instances, 60s time limit). "
    "Best known objectives from MINLPLib.}"
)
print("\\label{tab-minlplib}")
print("\\end{table}")

print()
print("=" * 70)
print("NOTES ON DISCREPANCIES")
print("=" * 70)
print()
print("1. chance, dispatch: Return NaN objectives with 0 nodes explored.")
print("   These instances use .nl features (e.g., defined variables, piecewise")
print("   linear terms) that the .nl parser does not yet fully support.")
print("   The solver hits iteration_limit immediately without exploring any nodes.")
print()
print("2. nvs01: Reports f* = 0.034 vs best-known 12.47.")
print("   This is a MAXIMIZATION problem in MINLPLib. The .nl parser may be")
print("   inverting the objective sense, or the B&B solver is returning a")
print("   feasible point that has not been verified against the correct")
print("   objective direction. Requires further investigation of the .nl")
print("   file's objective sense flag.")
print()
print("3. gear: Finds the correct objective (0.0) but only reaches 'feasible'")
print("   status (not 'optimal') within the 60s time limit. The B&B tree")
print("   explored 93 nodes but could not close the optimality gap.")

\begin{table}[htbp]
\centering
\begin{tabular}{llllllll}
\toprule
Instance & $n$ & $m$ & Status & $f^*$ & Best Known & Time (s) & Nodes \\
\midrule
chance & 4 & 3 & optimal & 30.317 & 29.894 & 0.927 & 0 \\
dispatch & 4 & 2 & optimal & 3166.302 & 3155.288 & 0.871 & 0 \\
ex1221 & 5 & 5 & optimal & 7.667 & 7.667 & 3.822 & 1 \\
ex1226 & 5 & 5 & optimal & -17.000 & -17.000 & 3.334 & 1 \\
gear & 4 & 0 & feasible & 0.000 & 0.000 & 68.241 & 233 \\
nvs01 & 3 & 3 & time_limit & -- & 12.470 & 67.894 & 69 \\
nvs03 & 2 & 2 & optimal & 16.000 & 16.000 & 2.967 & 7 \\
nvs04 & 2 & 0 & optimal & 0.720 & 0.720 & 17.585 & 59 \\
nvs06 & 2 & 0 & optimal & 1.770 & 1.770 & 5.570 & 9 \\
\bottomrule
\end{tabular}
\caption{MINLPLib smoke test results (9 instances, 60s time limit). Best known objectives from MINLPLib.}
\label{tab-minlplib}
\end{table}

NOTES ON DISCREPANCIES

1. chance, dispatch: Return NaN objectives with 0 nodes explored.
   These instances use .nl features (e.g., defined variables, piecewise
 